# Assignment 3 - Q5: Time-Lapse Microscopy Visualization Story

## Domain: Neuroscience - Calcium Imaging of Neural Activity

In [78]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import holoviews as hv
from holoviews.operation.datashader import rasterize
import hvplot.xarray # noqa
import panel as pn
import fsspec

pn.extension('tabulator')
hv.extension('bokeh')

In [79]:
DATA_URL = 'https://datasets.holoviz.org/miniscope/v1/real_miniscope_uint8.zarr/'
DATA_DIR = Path('./data')
DATA_FILENAME = Path(DATA_URL).name
DATA_PATH = DATA_DIR / DATA_FILENAME

print(f'Local Data Path: {DATA_PATH}')

Local Data Path: data\real_miniscope_uint8.zarr


In [80]:

# DATA_DIR.mkdir(parents=True, exist_ok=True)

# # Download the data
# print(f'Downloading data to: {DATA_PATH}')
# ds_remote = xr.open_dataset(
#     fsspec.get_mapper(DATA_URL), engine='zarr', chunks={}
# )
# ds_remote.to_zarr(str(DATA_PATH))  # Save locally
# print(f'Dataset downloaded to: {DATA_PATH}')

In [81]:
# Open the dataset from the local copy
ds = xr.open_dataset(
    DATA_PATH,
    engine='zarr',
    chunks={'frame': 400, 'height': -1, 'width': -1}  # Chunk by frames
)

# Display the dataset structure
ds

<xarray.Dataset> Size: 722MB
Dimensions:   (frame: 2000, height: 480, width: 752)
Coordinates:
  * frame     (frame) int64 16kB 0 1 2 3 4 5 6 ... 1994 1995 1996 1997 1998 1999
  * height    (height) int64 4kB 0 1 2 3 4 5 6 7 ... 473 474 475 476 477 478 479
  * width     (width) int64 6kB 0 1 2 3 4 5 6 7 ... 745 746 747 748 749 750 751
Data variables:
    varr_ref  (frame, height, width) uint8 722MB dask.array<chunksize=(400, 480, 752), meta=np.ndarray>

In [82]:
# Extract the data array
da = ds['varr_ref']

# Display basic information
print("="*60)
print("DATASET OVERVIEW")
print("="*60)
print(f"Data Array Shape: {da.shape}")
print(f"Data Type: {da.dtype}")
print(f"Total Size: {da.nbytes / (1024**2):.2f} MB")
print(f"\nDimensions:")
print(f"  - Frames (time points): {da.sizes['frame']}")
print(f"  - Height (pixels): {da.sizes['height']}")
print(f"  - Width (pixels): {da.sizes['width']}")
print(f"\nIntensity Range:")
print(f"  - Min: {da.min().values}")
print(f"  - Max: {da.max().values}")
print(f"  - Data type allows: 0-255 (8-bit unsigned integer)")
print("="*60)

# da is the 3d array 
da

DATASET OVERVIEW
Data Array Shape: (2000, 480, 752)
Data Type: uint8
Total Size: 688.48 MB

Dimensions:
  - Frames (time points): 2000
  - Height (pixels): 480
  - Width (pixels): 752

Intensity Range:
  - Min: 0
  - Max: 40
  - Data type allows: 0-255 (8-bit unsigned integer)
  - Max: 40
  - Data type allows: 0-255 (8-bit unsigned integer)


<xarray.DataArray 'varr_ref' (frame: 2000, height: 480, width: 752)> Size: 722MB
dask.array<open_dataset-varr_ref, shape=(2000, 480, 752), dtype=uint8, chunksize=(400, 480, 752), chunktype=numpy.ndarray>
Coordinates:
  * frame    (frame) int64 16kB 0 1 2 3 4 5 6 ... 1994 1995 1996 1997 1998 1999
  * height   (height) int64 4kB 0 1 2 3 4 5 6 7 ... 473 474 475 476 477 478 479
  * width    (width) int64 6kB 0 1 2 3 4 5 6 7 ... 745 746 747 748 749 750 751

In [83]:
def plot_image(frame):
    return hv.Image(da.sel(frame=frame), kdims=["width", "height"]).opts(
        title=f'Frame = {frame}',
        frame_height=da.sizes['height'],
        frame_width=da.sizes['width'],
        cmap='Viridis',
        clim=(0, 20),
        colorbar=True,
        tools=['hover', 'crosshair'],
        toolbar='right',
        apply_hard_bounds=True,
        scalebar=True,
        scalebar_unit=("µm", "m"),  # Each pixel is about 1 µm in this dataset
        scalebar_opts={
            'background_fill_alpha': 0.5,
            'border_line_color': None,
            'bar_length': 0.10,
        }
    )

In [84]:
frame_player = pn.widgets.Player(
    length=len(da.coords['frame']),
    interval=100,  # Inter-frame-interval in milliseconds
    value=20,     # Arbitrary starting frame
    show_loop_controls=False,
    align='center',
    scale_buttons=0.9,
    sizing_mode='stretch_width',
    show_value=True,
    value_align='center',
    visible_buttons=['slower', 'previous', 'pause', 'play', 'next', 'faster'],
)

In [85]:
# Compute the maximum projection over time
max_proj_time = da.max('frame').compute().astype(np.float32)

# Create the maximum projection image
img_max_proj_time = hv.Image(
    max_proj_time, ['width', 'height'], label='Max Over Time'
).opts(
    cmap='magma',
)

# Opacity slider for the overlay
opacity_slider = pn.widgets.FloatSlider(
    start=0, end=1, step=0.1, value=0.3, name='Opacity', align='center', sizing_mode='stretch_width'
)

# Link the slider value to the overlay's alpha (opacity)
opacity_slider.jslink(img_max_proj_time, value='glyph.global_alpha')

Link(args={}, bidirectional=False, code=None, name='Link08147', properties={'value': 'glyph.global_alpha'})

In [86]:
# Options for the side views
side_view_opts = dict(
    cmap='greys_r',
    tools=['crosshair', 'hover'],
    axiswise=True,
    apply_hard_bounds=True,
    colorbar=False,
    toolbar=None,
)

side_view_width = 175

# Top view: mean over height
top_data = da.mean('height').compute()
top_view = rasterize(
    hv.Image(top_data, kdims=['width', 'frame']).opts(
        frame_height=side_view_width,
        frame_width=da.sizes['width'],
        title='Top Side View',
        xaxis='top',
        **side_view_opts
    )
)

# Right view: mean over width
right_data = da.mean('width').compute()
right_view = rasterize(
    hv.Image(right_data, kdims=['frame', 'height']).opts(
        frame_height=da.sizes['height'],
        title='Right Side View',
        yaxis='right',
        frame_width=side_view_width,
        **side_view_opts
    )
)


In [87]:
def plot_hline(frame, x_range, y_range):
    if x_range is None:
        x_range = [int(da.width[0].values), int(da.width[-1].values)]
    return hv.Segments((x_range[0], frame, x_range[1], frame)).opts(axiswise=True)

def plot_vline(frame, x_range, y_range):
    if y_range is None:
        y_range = [int(da.height[0].values), int(da.height[-1].values)]
    return hv.Segments((frame, y_range[0], frame, y_range[1])).opts(axiswise=True)

line_opts = dict(color='red', line_width=3, line_alpha=0.4, line_dash='dashed')
xyrange_stream = hv.streams.RangeXY(source=main_view)
dmap_hline = hv.DynamicMap(pn.bind(plot_hline, frame_player), streams=[xyrange_stream]).opts(**line_opts, **side_view_opts)
dmap_vline = hv.DynamicMap(pn.bind(plot_vline, frame_player), streams=[xyrange_stream]).opts(**line_opts, **side_view_opts)

# Overlay the frame indicators on the side views
top_view_overlay = (top_view * dmap_hline).opts(axiswise=True)
right_view_overlay = (right_view * dmap_vline).opts(axiswise=True)


In [88]:
# Overlay the maximum projection on the main view
main_view_overlay = main_view * img_max_proj_time

# Arrange the main view and right side view horizontally
main_and_right_layout = pn.Row(main_view_overlay, right_view_overlay)

# Wrap the player and slider in collapsble Card widgets
player_layout = pn.Card(
    frame_player,
    title='Playback',
    sizing_mode='stretch_width',
    margin=(0, 0, 20, 0),
)

opacity_slider_layout = pn.Card(
    opacity_slider,
    title='Max Projection Overlay',
    sizing_mode='stretch_width',
    margin=(0, 0, 20, 0),
)

# Combine the controls
controls = pn.Column(
    player_layout,
    opacity_slider_layout,
    align='center',
    width=350,
)

# Assemble the full application layout
intermediate_app = pn.Row(
    controls,
    pn.Column(
        top_view_overlay,
        main_and_right_layout,
    )
)

In [89]:
from holonote.annotate import Annotator
from holonote.app import PanelWidgets, AnnotatorTable
from holonote.annotate.connector import SQLiteDB

# Initialize the annotator for 'height' and 'width' dimensions
annotator = Annotator(
    {'height': float, 'width': float},
    fields=['type'],  # Additional field to categorize annotations
    groupby='type',   # Group annotations to enable color-coding and batch actions
    connector=SQLiteDB(filename=':memory:'),
)

In [90]:
# OPTIONAL: Define colors for annotation 'types' that we might expect to make
color_dim = hv.dim('type').categorize(
    categories={
        'A': 'red',
        'B': 'orange',
        'C': 'cyan',
    },
    default='grey',
)

# Style the annotations
annotator.style.color = color_dim # apply custom colors if we made them
annotator.style.alpha = 0.3

In [91]:
# Create annotation widgets
panel_widgets = PanelWidgets(annotator)
table_widget = AnnotatorTable(
    annotator,
    tabulator_kwargs={
        'sizing_mode': 'stretch_width',
        'theme': 'midnight',
        'layout': 'fit_columns',
        'sortable': False,
        'stylesheets': [':host .tabulator {font-size: 9px;}'],
    },
)

In [92]:
# Options for the timeseries plot
curve_opts = dict(
    height=300,
    width=da.sizes['width'] + side_view_width + 225, # align to main_layout width
    show_legend=False,
    xlabel='Frame',
    tools=['hover'],
    line_alpha=0.5,
    framewise=True,
    axiswise=True,
)

# Options for the timeseries' frame indicator
vline_opts = dict(color='grey', line_width=4, alpha=0.5)

# Function to plot timeseries when annotations change
def plot_ts(event):
    curves = {}
    df = annotator.df
    for idx, row in df.iterrows():
        h1, h2, w1, w2 = row[['start[height]', 'end[height]', 'start[width]', 'end[width]']]
        da_sel = da.sel(height=slice(h1, h2), width=slice(w1, w2))
        mean_ts = da_sel.mean(['height', 'width'])
        group = f'G_{row["type"]}'
        label = f'L_{idx[:6]}'
        curve = hv.Curve(mean_ts, group=group, label=label)
        curve = curve.opts(
            subcoordinate_y=True,
            color=panel_widgets.colormap[row['type']],
            **curve_opts,
        )
        curves[(group, label)] = curve
    time_series.object = (vline * hv.Overlay(curves, kdims=['curve'])).opts(
        hv.opts.Curve(xlim=(frames[0], frames[-1])),
    )

# Function to create a vertical line indicating the current frame
def plot_frame_indicator_line(value):
    if value:
        return hv.VSpans((value, value)).opts(
            axiswise=True, framewise=True, **vline_opts
        )

# Get the range of frames
frames = da.coords['frame'].values

# Create a DynamicMap for the frame indicator line
vline = hv.DynamicMap(pn.bind(plot_frame_indicator_line, frame_player)).opts(
    hv.opts.VLine(**vline_opts)
)

# Initialize the timeseries pane
time_series = pn.pane.HoloViews(
    vline * hv.Curve([]).opts(
        xlim=(frames[0], frames[-1]),
        title='Create an annotation in the image',
        **curve_opts,
    )
)

# Connect the annotation events to the plotting function
annotator.on_event(plot_ts)

In [93]:
annotator.define_annotations(
    annotations_df,
    width=("x1", "x2"),
    height=("y1", "y2"),
    type='type',
)

In [94]:
# Overlay the annotator on the main view
main_view_overlay_anno = (main_view_overlay * annotator).opts(axiswise=True)
# Overlay the annotator on the side views
top_view_overlay_anno = (top_view_overlay * annotator).opts(axiswise=True)
right_view_overlay_anno = (right_view_overlay * annotator).opts(axiswise=True)

# Combine annotation controls and table into a Card
annotator_widgets = pn.Card(
    pn.Column(panel_widgets, table_widget),
    title='Annotator',
    sizing_mode='stretch_width',
    margin=(0, 0, 20, 0),
    collapsed=False,
)

# Add the annotator widgets to the controls
#controls.append(annotator_widgets)

# Create a fresh controls column instead of appending to the existing one
controls_advanced = pn.Column(
    player_layout,
    opacity_slider_layout,
    annotator_widgets,
    align='center',
    width=350,
)

# Create the main content layout
main_layout_anno = pn.Column(
    top_view_overlay_anno,
    pn.Row(main_view_overlay_anno, right_view_overlay_anno),
    time_series,
)

# Assemble the final application layout
advanced_app = pn.Row(
    controls_advanced,  # Fresh controls on the left
    main_layout_anno,   # Main content on the right
    align='start',
)

In [95]:
advanced_app

BokehModel(combine_events=True, render_bundle={'docs_json': {'48563787-207e-46f7-be1f-c9a06f33f270': {'version…